In [14]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy import ndimage
import re

In [18]:
BASE_DIR = r'Classification'
PATCHES_DIR = os.path.join(BASE_DIR, '../', 'rgb-complete_patches')
# WEIGHTS_PATH = os.path.join('../', 'field_model_pytorch.pth') #this path is for when we are in the 'src' directory
WEIGHTS_PATH = os.path.join('Classification', 'field_model_pytorch.pth') #this path is for when we are in the 'Field patch extraction' directory
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

THRESHOLD = 0.3
PATCH_SIZE = 224

# Text File Outputs
FIELD_TXT = os.path.join(BASE_DIR, "field list", "field_list.txt")
NON_FIELD_TXT = os.path.join(BASE_DIR, "field list", "non_field_list.txt")

In [16]:
def extract_coords(filename):
    parts = re.findall(r'\d+', filename)
    if len(parts) >= 2:
        return int(parts[-2]), int(parts[-1])
    return None

# --- 2. DL + NON-DL LOGIC FUNCTION ---
def run_classification_pipeline():
    # Load Model
    model = models.mobilenet_v2()
    model.classifier[1] = nn.Sequential(nn.Linear(model.last_channel, 1), nn.Sigmoid())
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.to(DEVICE).eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Scan files and build grid
    files = [f for f in os.listdir(PATCHES_DIR) if f.lower().endswith(('.png', '.tif', '.jpg'))]
    max_r, max_c = 0, 0
    patch_meta = []

    print(f"Running DL Inference on {len(files)} patches...")
    for fname in files:
        coords = extract_coords(fname)
        if coords:
            r, c = coords
            max_r, max_c = max(max_r, r), max(max_c, c)
            
            # DL Inference
            img = Image.open(os.path.join(PATCHES_DIR, fname)).convert('RGB')
            img_t = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                prob = model(img_t).item()
            
            # Initial DL Label (1 for field, 0 for non)
            label = 1 if prob < THRESHOLD else 0
            patch_meta.append({'name': fname, 'r': r, 'c': c, 'dl_label': label})

    # Create Grid for Non-DL Logic
    grid = np.zeros((max_r + 1, max_c + 1), dtype=int)
    name_grid = np.empty((max_r + 1, max_c + 1), dtype=object)
    for p in patch_meta:
        grid[p['r'], p['c']] = p['dl_label']
        name_grid[p['r'], p['c']] = p['name']

    print("Applying Non-DL Spatial Logic...")
    # Hole Filling
    refined_grid = ndimage.binary_fill_holes(grid).astype(int)
    # Island Removal (Keep only largest landmass)
    label_im, nb_labels = ndimage.label(refined_grid)
    if nb_labels > 1:
        sizes = ndimage.sum(refined_grid, label_im, range(nb_labels + 1))
        refined_grid = (label_im == np.argmax(sizes)).astype(int)

    # Write to Text Files
    print(f"Saving results to text files...")
    with open(FIELD_TXT, 'w') as f_out, open(NON_FIELD_TXT, 'w') as nf_out:
        for r in range(max_r + 1):
            for c in range(max_c + 1):
                fname = name_grid[r, c]
                if fname:
                    if refined_grid[r, c] == 1:
                        f_out.write(f"{fname}\n")
                    else:
                        nf_out.write(f"{fname}\n")
    print("Inference and Logic stage complete.")

# --- 3. RECONSTRUCTION FUNCTION ---
def reconstruct_from_txt():
    print("Reading text files for reconstruction...")
    
    # Read files into lists
    with open(FIELD_TXT, 'r') as f: fields = [line.strip() for line in f]
    with open(NON_FIELD_TXT, 'r') as f: non_fields = [line.strip() for line in f]

    # Find total dimensions
    max_r, max_c = 0, 0
    for fname in fields + non_fields:
        coords = extract_coords(fname)
        if coords:
            max_r, max_c = max(max_r, coords[0]), max(max_c, coords[1])

    full_w, full_h = (max_c + 1) * PATCH_SIZE, (max_r + 1) * PATCH_SIZE
    field_canvas = Image.new('RGB', (full_w, full_h), (0, 0, 0))
    non_field_canvas = Image.new('RGB', (full_w, full_h), (0, 0, 0))

    print(f"Stitching {len(fields) + len(non_fields)} patches...")
    
    # Process Fields
    for fname in fields:
        coords = extract_coords(fname)
        img = Image.open(os.path.join(PATCHES_DIR, fname)).convert('RGB').resize((PATCH_SIZE, PATCH_SIZE))
        field_canvas.paste(img, (coords[1] * PATCH_SIZE, coords[0] * PATCH_SIZE))

    # Process Non-Fields
    for fname in non_fields:
        coords = extract_coords(fname)
        img = Image.open(os.path.join(PATCHES_DIR, fname)).convert('RGB').resize((PATCH_SIZE, PATCH_SIZE))
        non_field_canvas.paste(img, (coords[1] * PATCH_SIZE, coords[0] * PATCH_SIZE))

    # Save
    field_canvas.save(os.path.join(BASE_DIR, "FINAL_FIELD_MAP.tif"))
    non_field_canvas.save(os.path.join(BASE_DIR, "FINAL_NON_FIELD_MAP.tif"))
    print("Stitching complete. Maps saved to Base Directory.")


In [19]:
run_classification_pipeline()
reconstruct_from_txt()

Running DL Inference on 3339 patches...
Applying Non-DL Spatial Logic...
Saving results to text files...
Inference and Logic stage complete.
Reading text files for reconstruction...
Stitching 3339 patches...
Stitching complete. Maps saved to Base Directory.


In [12]:
reconstruct_from_txt()

Reading text files for reconstruction...


FileNotFoundError: [Errno 2] No such file or directory: 'Classification\\field_list.txt'